In [1]:
!nvdia-smi

/bin/bash: line 1: nvdia-smi: command not found


In [2]:
!pip install -q ultralytics roboflow wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 153.6 MB/s eta 0:00:00


In [3]:
!pip install -q roboflow

In [4]:
import os
import yaml
import torch
import shutil
from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU", torch.cuda.get_device_name(0))

CUDA Available: True
GPU NVIDIA A100-SXM4-40GB


In [6]:
rf = Roboflow(api_key="EoPYQvXslfcNxfiMxKcD")
project = rf.workspace("kelompok-18").project("construction-safety-monitor-mlpd4-storl")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to construction-safety-monitor-1 in yolov8:: 100%|██████████| 10345/10345 [00:01<00:00, 6690.69it/s] 


In [7]:
dataset.location

'/content/construction-safety-monitor-1'

In [8]:
!ls {dataset.location}

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [9]:
model = YOLO("yolov8s.pt")

In [10]:
results = model.train(

    data=f"{dataset.location}/data.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=16,

    device=0,

    workers=8,

    optimizer="AdamW",

    lr0=0.001,
    lrf=0.01,

    weight_decay=0.0005,

    dropout=0.1,

    momentum=0.937,

    cos_lr=True,

    pretrained=True,

    val=True,

    cache=True,

    amp=True,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,

    flipud=0.0,
    fliplr=0.5,

    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,

    erasing=0.4,

    close_mosaic=10,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    plots=True,

    project="/content/human_safety_monitoring",

    name="yolov8s_ppe_detection"
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/construction-safety-monitor-1/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_ppe_detection, nbs=64, nms=False, opset=None, optimize=False, opt

In [11]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 11,129,841 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1182.0±345.9 MB/s, size: 38.5 KB)
val: Scanning /content/construction-safety-monitor-1/valid/labels.cache... 1026 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1026/1026 358.6Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 36, len(boxes) = 4310. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 65/65 9.1it/s 7.1s
                   all       1026       4310      0.461      0.572      0.511      0.304
                 boots        439        931      0.853      0.922       0.93      0.697
   

In [12]:
results = model.predict(
    source="/content/hha.jpg",
    conf=0.25,
    save=True
)


image 1/1 /content/hha.jpg: 448x640 3 goggless, 3 helmets, 3 vests, 75.5ms
Speed: 2.6ms preprocess, 75.5ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict


In [13]:
from google.colab import files

files.download(
    "/content/human_safety_monitoring/yolov8s_ppe_detection/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>